# **SPIDER_ML_TASK2_BASE_ML**

# **CUSTOM RESNET FOR CIFAR-10(Level1)**



# **AIM:  Building a custom ResNet-style CNN for CIFAR-10.**


# **structure of this task :**



1.importing the neccessary libraries

2.importing the dataset

3.Building the residual block

4.Building the resnet(residual network)

5.Train

6.Evalutate

7.Save the outputs and plot them on graph










# **1.IMPORT OF LIBRARIES**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

print("all libraries imported succesfully")

# model building,data loading,plotting and evaluation part will be done by these libraries

: 

# **Residual block**

In [ ]:
# This is the heart of ResNet
# The block learns a transformation and adds the original input back
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # First convolution: learns features from the input
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        # Second convolution: learns deeper features
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut path: identity by default
        self.shortcut = nn.Sequential()

        # If shape changes, it will match dimensions using 1x1 convolution
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        # Main path
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # Adding shortcut path
        out = out + self.shortcut(x)

        # Final activation
        out = F.relu(out)
        return out


## Full ResNet Model
This model stacks residual blocks into stages.Global average pooling is used before the final classifier.
Channels increase from 64 to 128 to 256 as the image gets smaller.


In [ ]:
class ResNetCIFAR10(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # Initial number of channels entering the first residual stage
        self.in_channels = 64

        # First convolution layer for 3-channel RGB input
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)

        # Residual stages
        self.layer1 = self._make_layer(64, blocks=2, stride=1)
        self.layer2 = self._make_layer(128, blocks=2, stride=2)
        self.layer3 = self._make_layer(256, blocks=2, stride=2)

        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Final classifier
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []

        # First block in the stage may change size or channels
        layers.append(ResidualBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels

        # Remaining blocks keep the same shape
        for _ in range(1, blocks):
            layers.append(ResidualBlock(self.in_channels, out_channels, stride=1))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.gap(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

## Data Loading(CIFAR-10)
CIFAR-10 contains 32x32 RGB images in 10 classes.
We use data augmentation for training and normalization for stable learning.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4) ,
    # This adds small spatial variation. It makes the model more robust to small shifts in the image
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])
# This combines multiple preprocessing steps into one pipeline

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])
# Normalization makes the optimization process more stable by putting input values into a consistent scale

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

## Model Setup
Here i am selecting the device, creating the ResNet model, defining the loss function, and choosing the optimizer

In [ ]:
# Picking up GPU when available so training is faster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Building our custom ResNet model
model = ResNetCIFAR10(num_classes=10).to(device)

# Standard loss for 10-class classification(CrossEntropyLoss)
criterion = nn.CrossEntropyLoss()

# Adam usually gives a good first baseline
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training and Evaluation Time
The training loop updates model weights using backpropagation.
The evaluation function checks performance on the test set without changing weights.

In [ ]:
# Lists to store training history for plots later
train_losses = []
test_losses = []
train_accs = []
test_accs = []

def evaluate(model, loader):
    # Switching to evaluation mode
    model.eval()

    correct = 0
    total = 0
    running_loss = 0
    all_preds = []
    all_labels = []

    # No gradients needed during evaluation
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    acc = 100.0 * correct / total
    return avg_loss, acc, all_labels, all_preds

# Number of training epochs
num_epochs = 4

for epoch in range(num_epochs):
    # Switch to training mode
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100.0 * correct / total

    test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%"
    )

In [ ]:
print(device)

## Final Evaluation and Saving
We plot training curves, print the final classification report and confusion matrix, and save the trained model weights

In [ ]:
# Plotting training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label="Train Accuracy")
plt.plot(test_accs, label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.tight_layout()
plt.savefig("training_curves.png")
plt.show()

# Final evaluation on test set
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader)

print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_acc:.2f}%")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# Saving trained weights
torch.save(model.state_dict(), "resnet_cifar10.pth")
print("Model will be saved as  resnet_cifar10.pth")

## Conclusion
In this notebook, I implemented a custom ResNet-style CNN for CIFAR-10 using manual residual blocks and skip connections.
I trained the model, evaluated it with standard classification metrics and saved the final weights.